# 📊 Análise Exploratória da Cana-de-Açúcar
## 02 — Explorando Padrões, Tendências e Distribuições

---

| | |
|---|---|
| 📅 **Safras** | 2017/18 a 2025/26 |
| 📊 **Fonte** | CONAB — Levantamentos Oficiais |
| 🗂️ **Base** | cana_limpo.csv (levantamento mais recente por safra) |
| 🔍 **Notebook** | 02 — Análise Exploratória (EDA) |

---

> **Objetivo:** Investigar padrões, tendências e distribuições da produção
> canavieira brasileira — analisando evolução por safra, ranking de estados,
> relação entre área plantada e produção, e qualidade da cana (ATR).

---

### 📌 O que será explorado neste notebook

- 🌾 **Produção total** por safra (evolução histórica)
- 🏆 **Top estados produtores** (ranking por UF)
- 📐 **Área plantada vs. Produção** (relação e eficiência)
- 🍬 **Açúcar e Etanol** (distribuição dos derivados)
- 📈 **ATR médio** (qualidade da cana ao longo das safras)

---

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')



In [8]:
df = pd.read_csv('D:/Area de trabalho/analise-producao-cana-conab/cana_limpo.csv', encoding='utf-8')
print(f'Dataset carregado: {df.shape}')
df.head()

Dataset carregado: (177, 17)


,ano_agricola,dsc_safra_previsao,uf,produto,id_produto,dsc_levantamento,id_levantamento,area_plantada_mil_ha,producao_mil_t,producao_acucar_mil_t,producao_etanol_anidro_mil_l,producao_etanol_hidratado_mil_l,producao_etanol_total_mil_l,produtcao_atr_kg_t,produtividade_t_ha,etanol_mil_m3,regiao
0,2017/18,UNICA,AL,CANA DE ACUCAR,4238,LEVANT,99,303.8,13646.9,1064.9,217373.0,109529.0,326902.0,123.6,44.92,326.90,Nordeste
1,2017/18,UNICA,AM,CANA DE ACUCAR,4238,LEVANT,99,3.6,222.1,11.9,0.0,4845.0,4845.0,93.1,61.69,4.84,Norte
2,2017/18,UNICA,BA,CANA DE ACUCAR,4238,LEVANT,99,47.1,3539.7,160.1,74714.0,105931.0,180645.0,135.3,75.15,180.64,Nordeste
3,2017/18,UNICA,ES,CANA DE ACUCAR,4238,LEVANT,99,47.6,2380.7,126.8,77370.0,13282.0,90652.0,122.7,50.01,90.65,Sudeste
4,2017/18,UNICA,GO,CANA DE ACUCAR,4238,LEVANT,99,911.6,70622.0,2234.6,1060506.0,3445990.0,4506496.0,142.2,77.47,4506.50,Centro-Oeste


## 1. Estatísticas descritivas 

In [9]:
print('=== ESTATÍSTICAS DESCRITIVAS ===')
df[['producao_mil_t','area_plantada_mil_ha','producao_acucar_mil_t',
    'producao_etanol_total_mil_l','produtcao_atr_kg_t']].describe().round(2)

=== ESTATÍSTICAS DESCRITIVAS ===


,producao_mil_t,area_plantada_mil_ha,producao_acucar_mil_t,producao_etanol_total_mil_l,produtcao_atr_kg_t
count,177.00,177.00,177.00,177.00,177.00
mean,32829.21,435.88,1952.49,1486243.08,129.75
std,75126.81,951.34,5199.75,3116362.53,12.87
min,29.70,0.70,0.00,1628.80,61.20
25%,1947.50,29.30,50.90,100500.00,126.00
50%,4092.80,57.80,147.70,245698.00,131.50
75%,30953.10,475.40,1495.30,1172871.20,137.20
max,383409.50,4558.40,28261.90,16489386.00,150.30


## 2. Produção total por safra

In [10]:
prod_safra = df.groupby('ano_agricola').agg(
    producao_total=('producao_mil_t', 'sum'),
    area_total=('area_plantada_mil_ha', 'sum'),
    acucar_total=('producao_acucar_mil_t', 'sum'),
    etanol_total=('producao_etanol_total_mil_l', 'sum'),
    atr_medio=('produtcao_atr_kg_t', 'mean')
).reset_index().sort_values('ano_agricola', ascending=False)  # ← adicione isso

prod_safra['atr_medio'] = prod_safra['atr_medio'].round(1)
prod_safra['producao_total'] = prod_safra['producao_total'].round(1)
print('Produção por safra (levantamento final):')
print(prod_safra.to_string(index=False))

Produção por safra (levantamento final):
ano_agricola  producao_total  area_total  acucar_total  etanol_total  atr_medio
     2025/26        666445.3      8974.8       45018.4    26551411.5      128.4
     2024/25        687058.8      8853.8       44853.6    29336013.0      131.8
     2023/24        713214.3      8334.1       45678.8    29689543.7      129.2
     2022/23        610131.3      8288.6       37036.3    27365888.4      129.1
     2021/22        585179.5      8317.4       35049.1    26784830.2      132.0
     2020/21        652326.2      8620.4       41254.3    29746423.0      132.0
     2019/20        642717.7      8442.0       29795.7    34001617.8      130.7
     2018/19        620435.4      8589.3       29038.2    32351643.0      127.1
     2017/18        633261.9      8729.5       37865.9    27237654.0      127.8


## 3. Top estados produtores (total acumulado)

In [11]:
top_estados = df.groupby('uf').agg(
    producao_acumulada=('producao_mil_t','sum'),
    media_por_safra=('producao_mil_t','mean'),
    area_media=('area_plantada_mil_ha','mean'),
    atr_medio=('produtcao_atr_kg_t','mean')
).round(1).sort_values('producao_acumulada', ascending=False)

print('Top estados produtores (produção acumulada em mil t):')
print(top_estados.head(10).to_string())

Top estados produtores (produção acumulada em mil t):
    producao_acumulada  media_por_safra  area_media  atr_medio
uf                                                            
SP           3062627.8         340292.0      4330.8      138.8
GO            668476.6          74275.2       957.3      139.9
MG            649145.9          72127.3       897.8      139.5
MS            434190.5          48243.4       657.4      133.3
PR            312850.2          34761.1       525.1      136.5
AL            159828.2          17758.7       298.2      128.0
MT            152898.0          16988.7       209.5      140.1
PE            115789.3          12865.5       232.2      128.1
PB             60122.7           6680.3       121.8      131.9
BA             44001.2           4889.0        57.9      136.2


## 4. Análise por região

In [12]:
prod_regiao = df.groupby(['regiao','ano_agricola']).agg(
    producao=('producao_mil_t','sum'),
    area=('area_plantada_mil_ha','sum')
).reset_index()

prod_regiao_total = prod_regiao.groupby('regiao')['producao'].sum().sort_values(ascending=False)
print('Produção acumulada por região (mil t):')
print(prod_regiao_total)

print('\nParticipação %:')
pct = (prod_regiao_total / prod_regiao_total.sum() * 100).round(1)
print(pct)

Produção acumulada por região (mil t):
regiao
Sudeste         3749959.5
Centro-Oeste    1255565.1
Nordeste         458396.2
Sul              312994.8
Norte             33854.8
Name: producao, dtype: float64

Participação %:
regiao
Sudeste         64.5
Centro-Oeste    21.6
Nordeste         7.9
Sul              5.4
Norte            0.6
Name: producao, dtype: float64


## 5. Evolução da produtividade

In [13]:
prod_sp = df[df['uf']=='SP'][['ano_agricola','produtividade_t_ha','produtcao_atr_kg_t']]
print('Produtividade SP (ton/ha e ATR):')
print(prod_sp.to_string(index=False))

Produtividade SP (ton/ha e ATR):
ano_agricola  produtividade_t_ha  produtcao_atr_kg_t
     2017/18               76.61               136.3
     2018/19               75.21               139.5
     2019/20               79.64               139.1
     2020/21               79.72               147.0
     2021/22               71.60               144.3
     2022/23               75.44               133.6
     2023/24               93.72               135.4
     2024/25               80.11               139.1
     2025/26               75.77               134.9


## 6. Correlações

In [14]:
cols = ['area_plantada_mil_ha','producao_mil_t','producao_acucar_mil_t',
        'producao_etanol_total_mil_l','produtcao_atr_kg_t','produtividade_t_ha']
corr = df[cols].corr().round(2)
print('Matriz de correlação:')
print(corr)

Matriz de correlação:
                             area_plantada_mil_ha  producao_mil_t  \
area_plantada_mil_ha                         1.00            1.00   
producao_mil_t                               1.00            1.00   
producao_acucar_mil_t                        0.98            0.98   
producao_etanol_total_mil_l                  0.98            0.98   
produtcao_atr_kg_t                           0.27            0.27   
produtividade_t_ha                           0.29            0.31   

                             producao_acucar_mil_t  \
area_plantada_mil_ha                          0.98   
producao_mil_t                                0.98   
producao_acucar_mil_t                         1.00   
producao_etanol_total_mil_l                   0.93   
produtcao_atr_kg_t                            0.23   
produtividade_t_ha                            0.27   

                             producao_etanol_total_mil_l  produtcao_atr_kg_t  \
area_plantada_mil_ha               

---

### 💡 Interpretação da Matriz de Correlação

**Correlações muito fortes (≈ 1.00):**
- A área plantada e a produção total são praticamente equivalentes — estados que plantam mais, produzem mais, de forma linear e previsível.
- Açúcar e etanol seguem diretamente a produção total (0.98), confirmando que o volume colhido é o principal driver dos derivados.

**Correlações fracas (0.23 ~ 0.40):**
- O **ATR médio** (qualidade da cana) tem baixa correlação com volume e área — indicando que qualidade é determinada por fatores agronômicos e climáticos, não pelo tamanho da operação.
- A **produtividade (t/ha)** também é independente da escala — estados menores podem ser tão ou mais eficientes que os grandes produtores.

> 📌 **Insight:** Escala de produção e eficiência produtiva são dimensões distintas no setor canavieiro brasileiro. Estados com menor área podem apresentar produtividade e qualidade superiores à média nacional.

---

## 7. Análise de precisão dos levantamentos

In [17]:
df_all = pd.read_csv('D:/Area de trabalho/analise-producao-cana-conab/cana_todos_levantamentos.csv', encoding='utf-8')

# Pegar 1º levantamento de cada safra/estado
lev1 = df_all[df_all['id_levantamento'] == 1][['ano_agricola', 'uf', 'producao_mil_t']]\
       .rename(columns={'producao_mil_t': 'prod_1lev'})

# Pegar o levantamento MAIS RECENTE 
idx = df_all.groupby(['ano_agricola', 'uf'])['id_levantamento'].transform('max')
levF = df_all[df_all['id_levantamento'] == idx][['ano_agricola', 'uf', 'producao_mil_t']]\
       .rename(columns={'producao_mil_t': 'prod_final'})

comparativo = pd.merge(lev1, levF, on=['ano_agricola', 'uf'])
comparativo = comparativo[comparativo['prod_final'] > 0]
comparativo['erro_pct'] = ((comparativo['prod_final'] - comparativo['prod_1lev']) / comparativo['prod_final'] * 100).round(2)

# Top 10 geral
print('=== Top 10 Estados com maior variação ===')
print(comparativo.nlargest(10, 'erro_pct')[['ano_agricola', 'uf', 'prod_1lev', 'prod_final', 'erro_pct']].to_string(index=False))

# Filtro SP
print('\n=== Variação histórica — São Paulo (SP) ===')
sp = comparativo[comparativo['uf'] == 'SP'].sort_values('ano_agricola', ascending=False)
print(sp[['ano_agricola', 'prod_1lev', 'prod_final', 'erro_pct']].to_string(index=False))

=== Top 10 Estados com maior variação ===
ano_agricola uf  prod_1lev  prod_final  erro_pct
     2021/22 RJ     1240.5      1833.4     32.34
     2020/21 RS       20.7        29.7     30.30
     2018/19 ES     2403.6      3174.1     24.27
     2022/23 RN     2821.1      3662.3     22.97
     2023/24 PR    31686.6     38730.9     18.19
     2024/25 ES     2640.0      3219.9     18.01
     2022/23 PB     5997.2      7302.4     17.87
     2018/19 AL    13339.1     16201.8     17.67
     2018/19 RJ      879.5      1057.5     16.83
     2024/25 AM      292.9       351.0     16.55

=== Variação histórica — São Paulo (SP) ===
ano_agricola  prod_1lev  prod_final  erro_pct
     2025/26   332907.5    335312.8      0.72
     2024/25   355090.9    353547.4     -0.44
     2023/24   322020.6    383409.5     16.01
     2022/23   301377.1    312879.5      3.68
     2021/22   326749.6    298494.8     -9.47
     2020/21   337058.1    354288.4      4.86
     2019/20   323416.4    342614.3      5.60
     2

---

### 🔍 Análise de Acurácia dos Levantamentos CONAB

Comparativo entre o **1º Levantamento** (estimativa inicial da safra) e o **resultado mais recente disponível**, medindo o percentual de variação por estado.

---

#### 📊 Top 10 Estados com Maior Variação

Os estados com maior divergência entre estimativa inicial e resultado final concentram-se majoritariamente no **Norte e Nordeste** — regiões com menor histórico de dados e maior instabilidade climática, o que dificulta projeções precisas no início da safra.

> 📌 **Destaque:** RJ (2021/22) apresentou a maior variação (32,34%), seguido de RS (2020/21) com 30,30% — ambos estados com produção menor e mais suscetível a oscilações climáticas.

---

#### 🏆 São Paulo — Variação Histórica (2017/18 a 2025/26)

São Paulo, maior produtor nacional, apresenta variações consistentemente **baixas e controladas** ao longo das safras — o que evidencia a maturidade do setor e a precisão dos levantamentos para o estado.

| Comportamento | Interpretação |
|---|---|
| Variações entre -1% e +6% na maioria das safras | Alta previsibilidade produtiva |
| 2023/24 com 16,01% de variação | Possível impacto climático ou revisão de área |
| 2021/22 com -9,47% | Produção final abaixo da estimativa inicial |

> 📌 **Insight:** A baixa variação histórica de SP reforça sua relevância como **referência de estabilidade** para o setor canavieiro nacional — contrastando com estados emergentes que ainda desenvolvem sua capacidade de estimativa.

---